In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch 
from bayesian_torch.models.dnn_to_bnn import get_kl_loss
from bayesian_torch.layers.variational_layers import LinearReparameterization
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

print("Library Versions:")
print('numpy:',np.__version__)
print('pandas:',pd.__version__)
print('torch:',torch.__version__)

In [ ]:

n_epochs = 100
verbose_option = True

# Regression for Naval Plant Maintenance

Load dataset

In [ ]:
npm = pd.read_csv('navalplantmaintenance.csv',header=None)
npm_train, npm_test = train_test_split(npm,test_size=0.25,random_state=42)
npm_train_np = npm_train.to_numpy()
npm_test_np = npm_test.to_numpy()
x_train_np = npm_train_np[:,:16]
x_test_np = npm_test_np[:,:16]
y_train_np = npm_train_np[:,17]
y_test_np = npm_test_np[:,17]
x_mu = x_train_np.mean(axis=0)
x_sigma = x_train_np.std(axis=0)
y_mu=y_train_np.mean(axis=0)
y_sigma=y_train_np.std(axis=0)
def scale(x,x_mu,x_sigma):
  x_sigma += 1e-16 #This is to deal with constant or near-constant columns
  return (x-x_mu)/x_sigma
def unscale(x,x_mu,x_sigma):
  x_sigma += 1e-16 #This is to deal with constant or near-constant columns
  return x_sigma*x+x_mu
x_train_np_z = scale(x_train_np,x_mu,x_sigma)
y_train_np_z = scale(y_train_np,y_mu,y_sigma)
x_test_np_z = scale(x_test_np,x_mu,x_sigma)
y_test_np_z = scale(y_test_np,y_mu,y_sigma)
x_train_t_z = torch.FloatTensor(x_train_np_z)
y_train_t_z = torch.FloatTensor(y_train_np_z)
x_test_t_z = torch.FloatTensor(x_test_np_z)
y_test_t_z = torch.FloatTensor(y_test_np_z)

1. Using PyTorch, perform variational inference using a mean-field Gaussian variational distribution and reparamaterization layers for Gaussian heteroscedastic regression.

In [ ]:
def nlls(y, mu, std):
    return torch.square(y - mu)/(2.0*torch.square(std))+torch.log(std)

class nn(torch.nn.Module):
    def __init__(self, inputSize, hiddenSize, outputSize):
        super(nn, self).__init__()
        self.layer1 = #TODO: Call the PyTorch Linear Local Reparameterization layer constructor going from inputSize to hiddenSize
        self.layer2 = #TODO: Call the PyTorch Linear Local Reparameterization layer constructor going from hiddenSize to hiddenSize
        self.linear_mu = #TODO: Call the PyTorch Linear Local Reparameterization layer constructor going from hiddenSize to outputSize
        self.linear_sigma = #TODO: Call the PyTorch Linear Local Reparameterization layer constructor going from hiddenSize to outputSize

    def forward(self, x):
        h1 = torch.nn.functional.relu(self.layer1(x,return_kl=False))
        h2 = torch.nn.functional.relu(self.layer2(h1,return_kl=False))
        mu = self.linear_mu(h2,return_kl=False)
        sigma = torch.nn.functional.softplus(self.linear_sigma(h2,return_kl=False))
        return mu, sigma

In [ ]:
x_train_t_z

In [ ]:
model = nn(x_train_np_z.shape[1],25, 1)

optimizer = torch.optim.Adam(params=model.parameters(), lr=1e-3)

for i in range(n_epochs):
    mu, s = model(x_train_t_z)
    M = mu.shape[0]
    nll_loss = nlls(y_train_t_z, mu, s).mean()
    kl = get_kl_loss(model)
    loss = #TODO: Write the VI loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()  
    if verbose_option: print(i, loss)

2.  Compute the mean and standard deviation predictions for 20 MC sampled models

In [ ]:
mc_samples = 20
n_test_examples = y_test_np.shape[0]
y_test_mus_z = np.zeros([mc_samples,n_test_examples,1])
y_test_sigmas_z = np.zeros([mc_samples,n_test_examples,1])
for i in range(mc_samples):
    mu, s = #TODO: Get prediction for one sampled parameter vector
    y_test_mus_z[i] = #TODO: Get the predicted probabilities of the test examples given the sampled parameter vector
    y_test_sigmas_z[i] = #TODO: Get the entropy of the predicted probability distributions of the test examples given the sampled parameter vector

In [ ]:
y_test_mus_z

3. Compute the Mean Squared Error (MSE) for the Gaussian VI model with Local Reparameterzaton layers for the test data using 20 MC samples and Bayesian model averaging.

In [ ]:
y_test_mu_z = y_test_mus_z.mean(axis=0) #TODO: Compute the mean predictions using Bayesian model averaging
y_test_mu = unscale(y_test_mu_z,y_mu,y_sigma)
print('MSE:', mean_squared_error(y_test_np, y_test_mu))

In [ ]:
y_test_mu.shape

4. Compute the aleatoric uncertainty for each regression test example for the Laplace approximation model.

In [ ]:
y_test_mus = unscale(y_test_mus_z,y_mu,y_sigma)
y_test_sigmas = y_test_sigmas_z * y_sigma
y_test_aleatoric = #TODO: Compute the aleatoric uncertainties
y_test_epistemic = #TODO: Compute the epistemic uncertainties

In [ ]:
y_test_aleatoric

In [ ]:
y_test_epistemic

# Classification for Ship Detection


Load Ship Detection Dataset

In [ ]:
import torch 
from torch.utils.data import Dataset, DataLoader
from bayesian_torch.models.dnn_to_bnn import dnn_to_bnn, get_kl_loss
from bayesian_torch.layers.flipout_layers import LinearFlipout 
from torchvision.io import read_image
from torch.utils.data import random_split
from torchvision.transforms.functional import resize
from sklearn import preprocessing
import numpy as np
from pathlib import Path
import torchmetrics

ROOT_PATH = "shipsnet/shipsnet"
LR = 1e-4
IMG_SIZE = [80]

tensor_size = IMG_SIZE[0]**2 * 3

def max_scaling(image):
    image = image / 255.0
    image = torch.Tensor(image)
    image = resize(image, size=IMG_SIZE)
    return image

def normalize_img(image):
    means = torch.Tensor([[[105.0385]],[[108.1886]],[[ 94.9558]]])
    stds = torch.Tensor([[[48.4294]],[[40.0104]],[[38.6445]]])
    image = torch.Tensor(image)
    image = resize(image, size=IMG_SIZE)
    image = image - means
    image = image / stds
    return image

#https://pytorch.org/tutorials/beginner/basics/data_tutorial.html
class ShipDataset(Dataset):
    def __init__(self, root_path, transform = None):
        self.root_path = Path(root_path)
        self.files = list(self.root_path.rglob("*/*"))
        self.classes = list(set([int(entry.parts[-1]) for entry in self.root_path.rglob("*") if Path(entry).is_dir()]))
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        image = read_image(str(self.files[idx]))
        label = int(self.files[idx].parts[-2])
        if self.transform:
            image = self.transform(image)
        return image, label
    

full_dataset  = ShipDataset(ROOT_PATH, transform = normalize_img,)
n_classes = len(full_dataset.classes)
print("Length of full dataset: ", len(full_dataset), " - with " ,n_classes ," classes. ")
train_dataset, test_dataset = random_split(full_dataset, [0.8, 0.2])
print("Length of training dataset: ", len(train_dataset))
print("Length of Test dataset: ", len(test_dataset))

train_dataloader = DataLoader(train_dataset, batch_size=len(train_dataset))
test_dataloader = DataLoader(test_dataset, batch_size=len(test_dataset))

criterion = torch.nn.BCELoss(reduce='mean')
accuracy = torchmetrics.classification.BinaryAccuracy()


5. Using PyTorch, perform variational inference using a mean-field Gaussian variational distribution and flipout layers for non-linear binary classification

In [ ]:
class logistic(torch.nn.Module):
    def __init__(self, inputSize, hiddenSize, outputSize):
        super(logistic, self).__init__()
        self.layer1 = #TODO: Call the PyTorch Linear Flipout layer constructor going from inputSize to hiddenSize
        self.layer2 = #TODO: Call the PyTorch Linear Flipout layer constructor going from hiddenSize to hiddenSize
        self.layer3 = #TODO: Call the PyTorch Linear Flipout layer constructor going from hiddenSize to hiddenSize
        self.layer4 = #TODO: Call the PyTorch Linear Flipout layer constructor going from hiddenSize to hiddenSize
        self.p = #TODO: Call the PyTorch Linear Flipout layer constructor going from hiddenSize to outputSize

    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        x = torch.nn.functional.relu(self.layer1(x,return_kl=False))
        x = torch.nn.functional.relu(self.layer2(x,return_kl=False))
        x = torch.nn.functional.relu(self.layer3(x,return_kl=False))
        x = torch.nn.functional.relu(self.layer4(x,return_kl=False))
        x = self.p(x,return_kl=False)
        return torch.sigmoid(x)

In [ ]:
model = logistic(tensor_size, 50, 1)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(n_epochs):
    for data, label in train_dataloader:
        p = model(data)
        M = p.shape[0]
        nll_loss = criterion(p.squeeze(),label*1.0) 
        kl = get_kl_loss(model)
        loss = #TODO: Write the VI loss
        acc = accuracy(p.squeeze(), label)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if verbose_option: print(epoch, loss, acc, end="/r")
    print(epoch)

6.  Compute the predicted probabilities and entropy predictions for 20 MC sampled models

In [ ]:
mc_samples = 20
n_test_examples = len(test_dataset)
y_test_probs = np.zeros([mc_samples,n_test_examples,1])
y_test_entropies = np.zeros([mc_samples,n_test_examples,1])
for i in range(mc_samples):
    for data, label in test_dataloader:
        results = model(data).detach().numpy()
        y_test_probs[i] = #TODO: Get the predicted means for one sampled parameter vector
        y_test_entropies[i] += #TODO: Get the entropy of the predicted probability distributions of the test examples given the sampled parameter vector

7. Compute the Bayesian model averaging predictions for each classification test example for the Laplace approximation model.

In [ ]:
y_test_probs_avg = #TODO: Compute Bayesian model averaging predictions

In [ ]:
y_test_probs_avg

8. Compute the aleatoric uncertainty for each classification test example for the Gaussian VI model.

In [ ]:
y_test_aleatoric  = #TODO: Computer aleatoric uncertainty

In [ ]:
y_test_aleatoric

9. Compute the epistemic uncertainty for each classification test example for the Gaussian VI model.

In [ ]:
y_test_uncertainty = #TODO: Compute the total uncertainity
y_test_epistemic =  #TODO: Compute the epistemic uncertainty

In [ ]:
y_test_epistemic

In [ ]:
from sklearn.metrics import classification_report

for data, label in test_dataloader:
    print(classification_report(label.flatten(), y_test_probs_avg.round().flatten()))